# cybORG — GRPO training in a single Colab

This notebook trains a small instruct model to play either the **red** (attacker) or **blue** (SOC analyst) side of CybOrg using TRL's GRPO trainer. The exact same `reward_funcs` callback used here lives in `training/cyborg_grpo.py` and is unit-tested in `tests/test_rewards.py`.

The cells are intentionally short so a judge can re-run them in 10 minutes on a Colab T4.

In [ ]:
!pip install -q openenv-core trl transformers accelerate unsloth

In [ ]:
!git clone https://github.com/your-team/cyborg.git
%cd cyborg

In [ ]:
import sys
sys.path.insert(0, '.')
from cyborg_env.server.cyborg_env_environment import CybOrgEnvironment
from training.cyborg_grpo import RandomPolicy, HeuristicRedPolicy, rollout_episode

env = CybOrgEnvironment()
rand = RandomPolicy(seed=0)
heur = HeuristicRedPolicy()

import statistics
rand_rewards = [rollout_episode(env, seed=s, task='red_easy', difficulty='easy', role='red', policy=rand, tokenizer=None).episode_reward for s in range(32)]
heur_rewards = [rollout_episode(env, seed=s, task='red_easy', difficulty='easy', role='red', policy=heur, tokenizer=None).episode_reward for s in range(32)]
print(f'random   mean={statistics.mean(rand_rewards):.3f}  std={statistics.stdev(rand_rewards):.3f}')
print(f'heuristic mean={statistics.mean(heur_rewards):.3f}  std={statistics.stdev(heur_rewards):.3f}')

## Train red with GRPO
Uses `Qwen/Qwen2.5-0.5B-Instruct` so it fits in a free Colab T4. Bump the model and `--episodes` for stronger runs.

In [ ]:
!python training/cyborg_grpo.py --model Qwen/Qwen2.5-0.5B-Instruct --task red_easy --difficulty easy --role red --episodes 64 --group-size 4 --output-dir runs/red

## Plot the reward curve

In [ ]:
import matplotlib.pyplot as plt
import json, glob, os
log_paths = glob.glob('runs/red/checkpoint-*/trainer_state.json')
if log_paths:
    log = json.load(open(sorted(log_paths)[-1]))
    rewards = [h.get('reward', 0.0) for h in log.get('log_history', []) if 'reward' in h]
    plt.plot(rewards)
    plt.xlabel('training step')
    plt.ylabel('mean group reward')
    plt.title('CybOrg — red_easy reward curve')
    plt.savefig('assets/red_reward_curve.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('no logs found yet — train for more steps')